# - Feature Engineering


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv("../data/processed/cleaned_salary_data.csv")
df.head()

,work_year,experience_level,employment_type,job_title,employee_residence,remote_ratio,company_location,company_size,salary_in_usd
0,2020,MI,FT,Data Scientist,DE,0,DE,L,79833
1,2020,SE,FT,Machine Learning Scientist,JP,0,JP,S,260000
2,2020,SE,FT,Big Data Engineer,GB,50,GB,M,109024
3,2020,MI,FT,Product Data Analyst,HN,0,HN,S,20000
4,2020,SE,FT,Machine Learning Engineer,US,50,US,L,150000


In [3]:
features = df.columns.drop("salary_in_usd").tolist()
print(features)
target = "salary_in_usd"

['work_year', 'experience_level', 'employment_type', 'job_title', 'employee_residence', 'remote_ratio', 'company_location', 'company_size']


In [4]:
df_numerical= df[features].select_dtypes(include=["int64","float64"]).columns.tolist()
df_numerical
df_categorical= df[features].select_dtypes(include=["object"]).columns.tolist()
df_categorical


/tmp/ipykernel_110974/924700039.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_categorical= df[features].select_dtypes(include=["object"]).columns.tolist()


['experience_level',
 'employment_type',
 'job_title',
 'employee_residence',
 'company_location',
 'company_size']

# 5. One-Hot Encoding


In [5]:
pd.get_dummies(df,columns=df_categorical)

,work_year,remote_ratio,salary_in_usd,experience_level_EN,experience_level_EX,experience_level_MI,experience_level_SE,employment_type_CT,employment_type_FL,employment_type_FT,...,company_location_RU,company_location_SG,company_location_SI,company_location_TR,company_location_UA,company_location_US,company_location_VN,company_size_L,company_size_M,company_size_S
0,2020,0,79833,False,False,True,False,False,False,True,...,False,False,False,False,False,False,False,True,False,False
1,2020,0,260000,False,False,False,True,False,False,True,...,False,False,False,False,False,False,False,False,False,True
2,2020,50,109024,False,False,False,True,False,False,True,...,False,False,False,False,False,False,False,False,True,False
3,2020,0,20000,False,False,True,False,False,False,True,...,False,False,False,False,False,False,False,False,False,True
4,2020,50,150000,False,False,False,True,False,False,True,...,False,False,False,False,False,True,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
560,2022,100,154000,False,False,False,True,False,False,True,...,False,False,False,False,False,True,False,False,True,False
561,2022,100,126000,False,False,False,True,False,False,True,...,False,False,False,False,False,True,False,False,True,False
562,2022,0,129000,False,False,False,True,False,False,True,...,False,False,False,False,False,True,False,False,True,False
563,2022,100,150000,False,False,False,True,False,False,True,...,False,False,False,False,False,True,False,False,True,False


In [6]:
pd.get_dummies(
    df,
    columns=["experience_level"],
    drop_first=True
)

,work_year,employment_type,job_title,employee_residence,remote_ratio,company_location,company_size,salary_in_usd,experience_level_EX,experience_level_MI,experience_level_SE
0,2020,FT,Data Scientist,DE,0,DE,L,79833,False,True,False
1,2020,FT,Machine Learning Scientist,JP,0,JP,S,260000,False,False,True
2,2020,FT,Big Data Engineer,GB,50,GB,M,109024,False,False,True
3,2020,FT,Product Data Analyst,HN,0,HN,S,20000,False,True,False
4,2020,FT,Machine Learning Engineer,US,50,US,L,150000,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...
560,2022,FT,Data Engineer,US,100,US,M,154000,False,False,True
561,2022,FT,Data Engineer,US,100,US,M,126000,False,False,True
562,2022,FT,Data Analyst,US,0,US,M,129000,False,False,True
563,2022,FT,Data Analyst,US,100,US,M,150000,False,False,True


# SCALING -- to make it in same line not to get the distance here and there between -1 0 1 only.

In [7]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df_scalar=df.copy()
df_scalar[df_numerical] = scaler.fit_transform(df[df_numerical])

# PIPELINE

In [16]:
X = df.drop(columns=["salary_in_usd"])

y = df["salary_in_usd"]

numeric_columns = [
    "work_year",
    "remote_ratio"
]

categorical_columns = [
    "experience_level",
    "employment_type",
    "job_title",
    "employee_residence",
    "company_location",
    "company_size"
]

# Previous pipeline error:
# `experience_level` was missing from categorical_columns.
# Because remainder="passthrough", sklearn passed raw text values like EN, MI, SE, EX into the final matrix.
# Linear Regression needs numeric input, so every text column must be encoded by OneHotEncoder.

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_columns
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_columns
        )
    ],
    # Use drop so no raw text column can accidentally pass into the model matrix.
    remainder="drop"
)

## Transform Features

Now we fit the preprocessing pipeline and transform `X` into a numeric feature matrix.

Expected result:

- numeric columns are scaled
- categorical columns are one-hot encoded
- no raw text columns remain

In [23]:
handled_columns = numeric_columns + categorical_columns
unhandled_columns = sorted(set(X.columns) - set(handled_columns))

print(f"Unhandled columns: {unhandled_columns}")

if unhandled_columns:
    raise ValueError(
        f"These columns are not assigned to preprocessing: {unhandled_columns}"
    )

X_transformed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

print(f"Original X shape: {X.shape}")
print(f"Transformed X shape: {X_transformed.shape}")
print(f"Number of generated feature names: {len(feature_names)}")

feature_names[:20]

Unhandled columns: []
Original X shape: (565, 8)
Transformed X shape: (565, 170)
Number of generated feature names: 170


array(['num__work_year', 'num__remote_ratio', 'cat__experience_level_EN',
       'cat__experience_level_EX', 'cat__experience_level_MI',
       'cat__experience_level_SE', 'cat__employment_type_CT',
       'cat__employment_type_FL', 'cat__employment_type_FT',
       'cat__employment_type_PT',
       'cat__job_title_3D Computer Vision Researcher',
       'cat__job_title_AI Scientist', 'cat__job_title_Analytics Engineer',
       'cat__job_title_Applied Data Scientist',
       'cat__job_title_Applied Machine Learning Scientist',
       'cat__job_title_BI Data Analyst',
       'cat__job_title_Big Data Architect',
       'cat__job_title_Big Data Engineer',
       'cat__job_title_Business Data Analyst',
       'cat__job_title_Cloud Data Engineer'], dtype=object)

In [25]:
# Optional inspection: convert a small sample of the transformed matrix into a dataframe.
# This is only for understanding; for modeling, we can use X_transformed directly.

sample_transformed_df = pd.DataFrame(
    X_transformed[:5].toarray() if hasattr(X_transformed, "toarray") else X_transformed[:5],
    columns=feature_names
)

sample_transformed_df.head()

,num__work_year,num__remote_ratio,cat__experience_level_EN,cat__experience_level_EX,cat__experience_level_MI,cat__experience_level_SE,cat__employment_type_CT,cat__employment_type_FL,cat__employment_type_FT,cat__employment_type_PT,...,cat__company_location_RU,cat__company_location_SG,cat__company_location_SI,cat__company_location_TR,cat__company_location_UA,cat__company_location_US,cat__company_location_VN,cat__company_size_L,cat__company_size_M,cat__company_size_S
0,-1.956361,-1.710815,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,-1.956361,-1.710815,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,-1.956361,-0.487257,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,-1.956361,-1.710815,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-1.956361,-0.487257,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
